# 07 - Model Training

Objective

This notebook prepares the modeling datasets, trains baseline and tuned candidate models, and selects the candidate final model for downstream evaluation.

The workflow preserves temporal ordering, prevents target leakage, compares Logistic Regression, Random Forest, and XGBoost, and persists modeling checkpoints for the evaluation and explainability notebooks.

Holdout testing, calibration analysis, fitted-model artifact saving, and model explainability are performed in Notebooks 08 and 09.

#### Load project configuration


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")


#### Load and validate the feature dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting


In [0]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T



FEATURE_TABLE = cfg.FEATURES_TABLE
TARGET_COLUMN = cfg.TARGET_COLUMN
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook (06) before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = cfg.MODEL_TRAINING_REQUIRED_COLUMNS

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

#### Date range and target distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.


In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

#### Create chronological train, validation, and test splits

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.


In [0]:
TRAIN_END_DATE = cfg.TRAIN_END_DATE
VALIDATION_START_DATE = cfg.VALIDATION_START_DATE
VALIDATION_END_DATE = cfg.VALIDATION_END_DATE
TEST_START_DATE = cfg.TEST_START_DATE

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

#### Split validation summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.


#### Engineer leakage-safe historical features

Historical performance features summarize prior delay behaviour for airlines, airports, and routes.

To prevent target leakage:

- Historical features for training records use only flights from earlier dates.
- The current record and later training outcomes are excluded.
- Validation and test mappings will be calculated from the training period only.
- A global training delay rate will be used when insufficient historical observations are available.

The first feature created is `AIRLINE_HIST_DELAY_RATE`, representing an airline's smoothed arrival-delay rate before the current flight date.


In [0]:
from pyspark.sql.window import Window


# Daily airline-level delay statistics within the training period
airline_daily_stats = (
    df_train
    .groupBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias("DAILY_DELAY_COUNT"),
    )
)

# Use only dates before the current flight date
airline_history_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

airline_daily_history = (
    airline_daily_stats
    .withColumn(
        "AIRLINE_PRIOR_FLIGHTS",
        F.sum("DAILY_FLIGHT_COUNT").over(airline_history_window),
    )
    .withColumn(
        "AIRLINE_PRIOR_DELAYS",
        F.sum("DAILY_DELAY_COUNT").over(airline_history_window),
    )
)

display(
    airline_daily_history
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "DAILY_FLIGHT_COUNT",
        "DAILY_DELAY_COUNT",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical airline delay rate

The `AIRLINE_HIST_DELAY_RATE` feature represents an airline's arrival-delay rate using only flights from earlier training dates.

A smoothed estimate is used to prevent unstable rates when an airline has limited prior observations. The overall training delay rate serves as the prior and as the fallback value for the first available date, when no earlier airline history exists.

The current date's outcomes are excluded from the calculation.


In [0]:
# Overall delay rate from the training period.
# This is used as the smoothing prior and first-date fallback.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col("ARR_DEL15").cast("double")).alias("GLOBAL_DELAY_RATE"))
    .first()["GLOBAL_DELAY_RATE"]
)

# Controls how strongly low-volume airline histories are pulled
# toward the global training delay rate.
SMOOTHING_STRENGTH = 100.0

airline_history_features = (
    airline_daily_history
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

# Join the leakage-safe airline history to every training flight.
df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Global training delay rate: {global_training_delay_rate:.6f}")
print(f"Smoothing strength: {SMOOTHING_STRENGTH:.0f}")
print(f"Training rows after join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical origin-airport delay rate

The `ORIGIN_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights departing from each origin airport.

For every training record, only flights from earlier dates at the same origin airport are included. The current date and all future records are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier airport history exists.


In [0]:
# Daily origin-airport delay statistics within the training period
origin_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ORIGIN_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ORIGIN_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
origin_history_window = (
    Window
    .partitionBy("ORIGIN")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

origin_history_features = (
    origin_daily_stats
    .withColumn(
        "ORIGIN_PRIOR_FLIGHTS",
        F.sum("DAILY_ORIGIN_FLIGHT_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_PRIOR_DELAYS",
        F.sum("DAILY_ORIGIN_DELAY_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

# Join origin history onto the training dataset that already contains
# AIRLINE_HIST_DELAY_RATE
df_train_hist = (
    df_train_hist
    .join(
        origin_history_features,
        on=["ORIGIN", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after origin join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical destination-airport delay rate

The `DEST_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights travelling to each destination airport.

For every training record, only flights from earlier dates with the same destination airport are included. The current date and all future observations are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier destination history exists.


In [0]:
# Daily destination-airport delay statistics within the training period
dest_daily_stats = (
    df_train
    .groupBy(
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_DEST_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_DEST_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
dest_history_window = (
    Window
    .partitionBy("DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

dest_history_features = (
    dest_daily_stats
    .withColumn(
        "DEST_PRIOR_FLIGHTS",
        F.sum("DAILY_DEST_FLIGHT_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_PRIOR_DELAYS",
        F.sum("DAILY_DEST_DELAY_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("DEST_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("DEST_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
    )
)

# Join destination history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        dest_history_features,
        on=["DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after destination join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical route delay rate

The `ROUTE_HIST_DELAY_RATE` feature represents the prior arrival-delay rate for each origin–destination route.

For every training record, only flights from earlier dates on the same route are included. The current date and all future observations are excluded to prevent target leakage.

Because some routes have limited historical volume, a smoothed estimate is used. The global training delay rate serves as the prior and as the fallback when no earlier route history exists.


In [0]:
# Daily route-level delay statistics within the training period
route_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ROUTE_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ROUTE_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
route_history_window = (
    Window
    .partitionBy("ORIGIN", "DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

route_history_features = (
    route_daily_stats
    .withColumn(
        "ROUTE_PRIOR_FLIGHTS",
        F.sum("DAILY_ROUTE_FLIGHT_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_PRIOR_DELAYS",
        F.sum("DAILY_ROUTE_DELAY_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
    )
)

# Join route history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after route join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

#### Apply training-only historical features to validation and test data

Historical mappings for airlines, origin airports, destination airports, and routes are calculated exclusively from the training period.

These fixed training-period mappings are then joined to the validation and test datasets. Neither validation nor test outcomes are used when calculating the historical rates.

For categories not observed during training, the global training delay rate is used as a fallback.


In [0]:
# -------------------------------------------------------
# Create historical mappings from training data only
# -------------------------------------------------------

airline_training_map = (
    df_train
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.count("*").alias("AIRLINE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "AIRLINE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.col("AIRLINE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("AIRLINE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

origin_training_map = (
    df_train
    .groupBy("ORIGIN")
    .agg(
        F.count("*").alias("ORIGIN_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ORIGIN_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.col("ORIGIN_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ORIGIN_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

dest_training_map = (
    df_train
    .groupBy("DEST")
    .agg(
        F.count("*").alias("DEST_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DEST_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.col("DEST_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("DEST_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "DEST_HIST_DELAY_RATE",
    )
)

route_training_map = (
    df_train
    .groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("ROUTE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ROUTE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.col("ROUTE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ROUTE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "ROUTE_HIST_DELAY_RATE",
    )
)


def attach_training_history(dataset: DataFrame) -> DataFrame:
    """Attach historical rates calculated exclusively from training data."""
    return (
        dataset
        .join(
            airline_training_map,
            on="OP_UNIQUE_CARRIER",
            how="left",
        )
        .join(
            origin_training_map,
            on="ORIGIN",
            how="left",
        )
        .join(
            dest_training_map,
            on="DEST",
            how="left",
        )
        .join(
            route_training_map,
            on=["ORIGIN", "DEST"],
            how="left",
        )
        .fillna(
            {
                "AIRLINE_HIST_DELAY_RATE": global_training_delay_rate,
                "ORIGIN_HIST_DELAY_RATE": global_training_delay_rate,
                "DEST_HIST_DELAY_RATE": global_training_delay_rate,
                "ROUTE_HIST_DELAY_RATE": global_training_delay_rate,
            }
        )
    )


df_validation_hist = attach_training_history(df_validation)
df_test_hist = attach_training_history(df_test)

print(
    f"Validation rows after historical joins: "
    f"{df_validation_hist.count():,}"
)
print(
    f"Test rows after historical joins: "
    f"{df_test_hist.count():,}"
)

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "AIRLINE_HIST_DELAY_RATE",
        "ORIGIN_HIST_DELAY_RATE",
        "DEST_HIST_DELAY_RATE",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .limit(20)
)

#### Validate historical performance features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns


In [0]:
HISTORICAL_RATE_COLUMNS = list(cfg.MODEL_HISTORICAL_RATE_COLUMNS)

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)


#### Encode categorical variables

Categorical and numerical predictors are converted into a model-ready sparse feature vector using Spark's `FeatureHasher`.

Feature hashing maps categorical values into a fixed-dimensional numerical representation without fitting and storing large category-indexing models. This approach avoids the Spark Connect ML model-cache limitation encountered with `StringIndexer` and `OneHotEncoder`.

The same deterministic transformation is applied to the training, validation, and test datasets. No validation or test outcomes are used during preprocessing.

Before applying the transformation, the preprocessing configuration is validated to ensure that all required model input columns are present across the chronological training, validation, and test datasets. The resulting hashed datasets retain only `FL_DATE`, the target variable, and the generated `features` vector required by the machine learning algorithms. Each transformed dataset is then validated to ensure that feature hashing has been applied successfully before model training begins.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from utils.model_training import (
    create_feature_hasher,
    hash_modeling_frame,
    prepare_hist_modeling_frame,
    validate_hist_modeling_frame,
    validate_feature_hasher,
)


CATEGORICAL_COLUMNS = list(cfg.MODEL_CATEGORICAL_COLUMNS)
NUMERICAL_COLUMNS = list(cfg.MODEL_NUMERICAL_COLUMNS)
MODEL_INPUT_COLUMNS = list(cfg.MODEL_INPUT_COLUMNS)
HASH_VECTOR_SIZE = cfg.HASH_VECTOR_SIZE

# Drop intermediate prior-count columns before hashing or checkpoint persistence.
df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

for dataframe_name, dataframe in {
    "df_train_hist": df_train_hist,
    "df_validation_hist": df_validation_hist,
    "df_test_hist": df_test_hist,
}.items():
    validate_hist_modeling_frame(dataframe, dataframe_name)

feature_hasher = create_feature_hasher()
validate_feature_hasher(feature_hasher)

df_train_hashed = hash_modeling_frame(df_train_hist, feature_hasher)
df_validation_hashed = hash_modeling_frame(df_validation_hist, feature_hasher)
df_test_hashed = hash_modeling_frame(df_test_hist, feature_hasher)

for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    validation_row_count = (
        dataframe
        .select("features")
        .limit(1)
        .count()
    )

    if validation_row_count == 0:
        raise ValueError(
            f"{dataframe_name} contains no rows."
        )

    print(
        f"{dataframe_name} created and validated successfully."
    )

print()
print("Feature-hashing preprocessing configured successfully.")
print(f"Categorical features: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical features: {len(NUMERICAL_COLUMNS)}")
print(f"Total raw predictors: {len(MODEL_INPUT_COLUMNS)}")
print(f"Hashed vector size: {HASH_VECTOR_SIZE:,}")
print(f"Target: {TARGET_COLUMN}")
print(f"Feature hasher inputs: {feature_hasher.getInputCols()}")

df_train_prepared = df_train_hashed
df_validation_prepared = df_validation_hashed
df_test_prepared = df_test_hashed


#### Apply Feature Hashing to Model Datasets

The configured Spark `FeatureHasher` is applied to the chronological training, validation, and test datasets as a scalable data-preparation step.

This deterministic transformation converts the selected numerical and categorical predictors into a fixed-size sparse `features` vector. The transformed DataFrames retain `FL_DATE` and the target variable so they can later be converted to SciPy sparse matrices for standard-Python model training.

Spark is used here only for distributed data preparation; Logistic Regression, Random Forest, and XGBoost are trained with standard Python libraries.


In [0]:
for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    missing_columns = sorted(
        set(cfg.MODEL_HASHED_OUTPUT_COLUMNS) - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            f"{missing_columns}"
        )

    print(
        f"{dataframe_name} ready for modeling with "
        f"{dataframe.count():,} rows."
    )


#### Validate the Prepared Sparse Feature Datasets

Feature hashing is deterministic and does not learn category mappings from validation or test outcomes. The same configured transformation is applied to all chronological datasets.

The resulting `features` vectors are validated together with `FL_DATE` and the target column. These vectors form the common input representation that will be converted to SciPy sparse matrices and reused across all three standard-Python algorithms.


In [0]:
print("Feature hashing applied successfully.")
print(f"Training rows: {df_train_hashed.count():,}")
print(f"Validation rows: {df_validation_hashed.count():,}")
print(f"Test rows: {df_test_hashed.count():,}")

display(
    df_train_hashed
    .select(
        *cfg.MODELING_JOIN_KEY_COLUMNS,
        "features",
        cfg.TARGET_COLUMN,
    )
    .limit(10)
)


## Standard-Python Candidate Modeling

Logistic Regression, Random Forest, and XGBoost are implemented using standard Python libraries rather than Spark ML. Spark remains responsible only for loading the large Delta tables, applying the leakage-safe feature engineering workflow, and creating bounded samples that can be safely collected by the Databricks Free Edition driver.

The three algorithms receive the same SciPy sparse feature matrices, chronological folds, validation observations, evaluation metrics, and selection policy. This shared design removes implementation differences that could otherwise make the comparison difficult to interpret.

The candidate estimators are:

- `sklearn.linear_model.LogisticRegression`
- `sklearn.ensemble.RandomForestClassifier`
- `xgboost.XGBClassifier`


In [0]:
try:
    import numpy as np
    import pandas as pd

    from scipy.sparse import csr_matrix
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        confusion_matrix,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.model_selection import ParameterGrid
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError(
        "Notebook 07 requires NumPy, SciPy, scikit-learn, and XGBoost. "
        "Run `%pip install xgboost==2.1.4 scipy scikit-learn`, restart "
        "Python once, and rerun the notebook from the beginning."
    ) from error

import ast
import time

from pyspark.sql import functions as F

RANDOM_SEED = cfg.RANDOM_SEED
LOCAL_TRAIN_MAX_ROWS = 50_000
LOCAL_VALIDATION_MAX_ROWS = 50_000

print("Standard-Python modeling libraries loaded successfully.")
print(f"Maximum local training rows per fold: {LOCAL_TRAIN_MAX_ROWS:,}")
print(
    "Maximum local validation rows per fold: "
    f"{LOCAL_VALIDATION_MAX_ROWS:,}"
)


## Reproducible Local-Matrix Utilities

The prepared Spark feature vectors are converted directly into SciPy compressed sparse row (CSR) matrices. The vectors are never expanded into a dense pandas table, which limits driver-memory usage while preserving the hashed feature representation.

Sampling is uniform and reproducible. It does not rebalance the validation data. Consequently, the validation samples retain the natural proportion of delayed and on-time flights.


In [0]:
def bounded_uniform_sample(
    dataframe,
    maximum_rows,
    *,
    seed,
):
    """Return a reproducible uniform sample with at most maximum_rows."""
    row_count = dataframe.count()

    if row_count == 0:
        raise ValueError("Cannot sample an empty Spark DataFrame.")

    if row_count <= maximum_rows:
        return dataframe

    sampling_fraction = min(
        1.0,
        (maximum_rows * 1.10) / row_count,
    )

    return (
        dataframe
        .sample(
            withReplacement=False,
            fraction=sampling_fraction,
            seed=seed,
        )
        .limit(maximum_rows)
    )


def spark_vectors_to_csr(dataframe):
    """Collect Spark vectors as a SciPy CSR matrix and NumPy labels."""
    rows = (
        dataframe
        .select("features", TARGET_COLUMN)
        .toLocalIterator()
    )

    data = []
    indices = []
    indptr = [0]
    labels = []
    feature_count = None

    for row in rows:
        vector = row["features"]
        feature_count = int(vector.size)

        if hasattr(vector, "indices"):
            indices.extend(int(index) for index in vector.indices)
            data.extend(float(value) for value in vector.values)
        else:
            dense_values = np.asarray(vector, dtype=np.float32)
            nonzero = np.flatnonzero(dense_values)
            indices.extend(nonzero.tolist())
            data.extend(dense_values[nonzero].tolist())

        indptr.append(len(data))
        labels.append(int(row[TARGET_COLUMN]))

    if not labels or feature_count is None:
        raise ValueError("The local modeling sample contains zero rows.")

    matrix = csr_matrix(
        (
            np.asarray(data, dtype=np.float32),
            np.asarray(indices, dtype=np.int32),
            np.asarray(indptr, dtype=np.int64),
        ),
        shape=(len(labels), feature_count),
        dtype=np.float32,
    )

    return matrix, np.asarray(labels, dtype=np.int8)


def evaluate_python_classifier(model, X_validation, y_validation):
    """Evaluate a fitted binary classifier using the shared metrics."""
    predictions = model.predict(X_validation).astype(np.int8)
    probabilities = model.predict_proba(X_validation)[:, 1]

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    delay_precision, delay_recall, delay_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )

    return {
        "ACCURACY": float(
            accuracy_score(y_validation, predictions)
        ),
        "PRECISION": float(weighted_precision),
        "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "ROC_AUC": float(
            roc_auc_score(y_validation, probabilities)
        ),
        "PR_AUC": float(
            average_precision_score(
                y_validation,
                probabilities,
            )
        ),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
    }


def positive_class_weight(labels):
    """Return the negative-to-positive ratio for XGBoost."""
    positive_count = int(np.sum(labels == 1))
    negative_count = int(np.sum(labels == 0))

    if positive_count == 0 or negative_count == 0:
        raise ValueError(
            "Both target classes must be present in every training fold."
        )

    return negative_count / positive_count


def confusion_matrix_table(y_true, y_predicted):
    """Return a clearly labeled binary confusion matrix."""
    matrix = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    )
    return pd.DataFrame(
        matrix,
        index=["Actual On Time (0)", "Actual Delayed (1)"],
        columns=["Predicted On Time (0)", "Predicted Delayed (1)"],
    )


def confusion_count_table(y_true, y_predicted):
    """Return explicit TN, FP, FN, and TP counts with definitions."""
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    ).ravel()

    return pd.DataFrame(
        [
            {
                "CONFUSION_TERM": "TN",
                "MEANING": "Actual on-time flight predicted as on time",
                "COUNT": int(tn),
            },
            {
                "CONFUSION_TERM": "FP",
                "MEANING": "Actual on-time flight predicted as delayed",
                "COUNT": int(fp),
            },
            {
                "CONFUSION_TERM": "FN",
                "MEANING": "Actual delayed flight predicted as on time",
                "COUNT": int(fn),
            },
            {
                "CONFUSION_TERM": "TP",
                "MEANING": "Actual delayed flight predicted as delayed",
                "COUNT": int(tp),
            },
        ]
    )


print("Shared local-matrix and evaluation utilities created.")


## Class-Imbalance Strategy

Delayed flights form the minority class. A classifier can therefore achieve high overall Accuracy by predicting most flights as on time while missing many actual delays.

Class imbalance is handled only within each training dataset:

- Logistic Regression uses `class_weight="balanced"`.
- Random Forest uses `class_weight="balanced_subsample"`.
- XGBoost uses `scale_pos_weight`, calculated as the number of on-time training observations divided by the number of delayed training observations.

Validation observations are not oversampled, undersampled, or synthetically generated. Keeping the validation distribution unchanged produces metrics that reflect realistic flight operations and prevents information from the validation period from influencing model training.

Delayed-flight Recall, delayed-flight F1-score, and PR AUC are reported in addition to overall and weighted metrics. These metrics expose minority-class performance that Accuracy alone may conceal.


## Metric Definitions and Delayed-Class Formulas

The target is binary: `0` represents an on-time flight and `1` represents a delayed flight. Metrics beginning with `DELAY_` evaluate class `1` specifically, whereas the unprefixed Precision, Recall, and F1 columns are weighted averages across both classes.

Let `TP` be delayed flights correctly predicted as delayed, `FN` be delayed flights incorrectly predicted as on time, and `FP` be on-time flights incorrectly predicted as delayed.

**Delayed-flight Recall**

`DELAY_RECALL = TP / (TP + FN)`

This answers: Of all flights that actually became delayed, what proportion did the model detect?

**Delayed-flight Precision**

`DELAY_PRECISION = TP / (TP + FP)`

This answers: Of all flights flagged as delayed, what proportion actually became delayed?

**Delayed-flight F1-score**

`DELAY_F1 = 2 * (DELAY_PRECISION * DELAY_RECALL) / (DELAY_PRECISION + DELAY_RECALL)`

This balances detecting delayed flights against avoiding excessive false delay alerts.

The result-table names have the following meanings:

- `ACCURACY`: proportion of all predictions that are correct.
- `PRECISION`: class-frequency-weighted Precision across classes `0` and `1`.
- `RECALL`: class-frequency-weighted Recall across classes `0` and `1`; in single-label classification it equals Accuracy and is not the delayed-class Recall.
- `F1_SCORE`: class-frequency-weighted F1 across classes `0` and `1`.
- `DELAY_PRECISION`, `DELAY_RECALL`, and `DELAY_F1`: metrics for delayed flights only.
- `ROC_AUC`: discrimination between on-time and delayed flights across thresholds.
- `PR_AUC`: probability-ranking quality for the minority delayed-flight class.

Delayed-flight Recall is emphasized because false negatives are actual delays that receive no warning. It is interpreted together with delayed-flight F1 and PR AUC so that a model is not rewarded merely for flagging nearly every flight as delayed.


## Shared Baseline Dataset

Before hyperparameter tuning, the three standard-Python algorithms are trained and evaluated using the same bounded chronological datasets. Training observations come only from the January–August development period. Validation observations come from the later September–October validation period.

The same feature matrix and labels are reused across all three algorithms, ensuring that baseline differences arise from the algorithms rather than from different samples.


In [0]:
baseline_training_df = bounded_uniform_sample(
    df_train_prepared.select("features", TARGET_COLUMN),
    LOCAL_TRAIN_MAX_ROWS,
    seed=RANDOM_SEED,
)
baseline_validation_df = bounded_uniform_sample(
    df_validation_prepared.select("features", TARGET_COLUMN),
    LOCAL_VALIDATION_MAX_ROWS,
    seed=RANDOM_SEED + 1,
)

X_baseline_train, y_baseline_train = spark_vectors_to_csr(
    baseline_training_df
)
X_baseline_validation, y_baseline_validation = (
    spark_vectors_to_csr(baseline_validation_df)
)

print(f"Shared baseline training rows: {X_baseline_train.shape[0]:,}")
print(
    "Shared baseline validation rows: "
    f"{X_baseline_validation.shape[0]:,}"
)
print(
    "Baseline training delayed-flight rate: "
    f"{np.mean(y_baseline_train):.4f}"
)
print(
    "Baseline validation delayed-flight rate: "
    f"{np.mean(y_baseline_validation):.4f}"
)


## Train and Evaluate Standard-Python Baselines

The baseline configurations provide an initial reference before tuning. Class-imbalance controls are enabled for every trainable candidate, while the majority-class baseline demonstrates why Accuracy alone is insufficient.


In [0]:
majority_class = int(
    np.bincount(y_baseline_train).argmax()
)
majority_predictions = np.full(
    y_baseline_validation.shape,
    majority_class,
    dtype=np.int8,
)

majority_weighted_precision, majority_weighted_recall, majority_weighted_f1, _ = (
    precision_recall_fscore_support(
        y_baseline_validation,
        majority_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_rows = [
    {
        "MODEL": "Majority Class Baseline",
        "ACCURACY": float(
            accuracy_score(
                y_baseline_validation,
                majority_predictions,
            )
        ),
        "PRECISION": float(majority_weighted_precision),
        "RECALL": float(majority_weighted_recall),
        "F1_SCORE": float(majority_weighted_f1),
        "ROC_AUC": None,
        "PR_AUC": None,
        "DELAY_PRECISION": 0.0,
        "DELAY_RECALL": 0.0,
        "DELAY_F1": 0.0,
    }
]

baseline_estimators = {
    "Logistic Regression (Baseline)": LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=500,
        random_state=RANDOM_SEED,
    ),
    "Random Forest (Baseline)": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "XGBoost (Baseline)": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.10,
        min_child_weight=1,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=positive_class_weight(y_baseline_train),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

baseline_predictions = {}

for model_name, estimator in baseline_estimators.items():
    estimator.fit(X_baseline_train, y_baseline_train)
    baseline_predictions[model_name] = (
        estimator.predict(X_baseline_validation).astype(np.int8)
    )
    metrics = evaluate_python_classifier(
        estimator,
        X_baseline_validation,
        y_baseline_validation,
    )
    baseline_rows.append(
        {
            "MODEL": model_name,
            **metrics,
        }
    )

model_comparison = spark.createDataFrame(baseline_rows)
display(model_comparison.orderBy("MODEL"))


## Baseline Confusion Matrices

The confusion matrices below show how each untuned algorithm classified the same chronological validation observations. Rows represent actual outcomes and columns represent predicted outcomes.

- The upper-left cell is the number of correctly identified on-time flights (true negatives).
- The upper-right cell is the number of on-time flights incorrectly flagged as delayed (false positives).
- The lower-left cell is the number of delayed flights missed by the model (false negatives).
- The lower-right cell is the number of delayed flights correctly identified (true positives).

For this project, the lower-left cell is especially important because it contains actual delays that would receive no operational warning.


In [0]:
for model_name, predictions in baseline_predictions.items():
    print(f"Baseline confusion matrix: {model_name}")
    display(
        confusion_matrix_table(
            y_baseline_validation,
            predictions,
        )
    )
    print(f"Baseline TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            y_baseline_validation,
            predictions,
        )
    )


## Baseline Comparison Interpretation

The majority-class baseline is included as a diagnostic reference rather than a viable operational model. Its Accuracy can appear strong because most flights are on time, but its delayed-flight Recall is zero.

The three machine-learning baselines use identical local training and validation matrices and explicitly account for class imbalance. Their relative performance therefore provides a fair preliminary comparison. Final selection is not based on this single validation period; it is based on average performance across the shared chronological cross-validation folds below.


## Unified Chronological Cross-Validation

One cross-validation workflow is used for Logistic Regression, Random Forest, and XGBoost.

The January–August development period is divided into four expanding-window folds:

| Fold | Training period | Validation period |
|---|---|---|
| 1 | January–April 2025 | May 2025 |
| 2 | January–May 2025 | June 2025 |
| 3 | January–June 2025 | July 2025 |
| 4 | January–July 2025 | August 2025 |

For each fold, the chronological date boundaries are applied before sampling. One reproducible uniform training sample and one reproducible uniform validation sample are then created and converted to local sparse matrices. These exact matrices are reused by every algorithm and every hyperparameter configuration.

This arrangement prevents future observations from entering earlier training periods, keeps validation distributions natural, and avoids algorithm-specific sampling differences.


In [0]:
TUNING_FOLDS = [
    {
        "train_end": "2025-04-30",
        "validation_start": "2025-05-01",
        "validation_end": "2025-05-31",
    },
    {
        "train_end": "2025-05-31",
        "validation_start": "2025-06-01",
        "validation_end": "2025-06-30",
    },
    {
        "train_end": "2025-06-30",
        "validation_start": "2025-07-01",
        "validation_end": "2025-07-31",
    },
    {
        "train_end": "2025-07-31",
        "validation_start": "2025-08-01",
        "validation_end": "2025-08-31",
    },
]

df_tuning_complete = (
    df_train_prepared
    .select("FL_DATE", "features", TARGET_COLUMN)
)

LOCAL_CV_FOLDS = []

for fold_number, fold in enumerate(TUNING_FOLDS, start=1):
    complete_training_fold = df_tuning_complete.filter(
        F.col("FL_DATE")
        <= F.to_date(F.lit(fold["train_end"]))
    )
    complete_validation_fold = df_tuning_complete.filter(
        F.col("FL_DATE").between(
            F.to_date(F.lit(fold["validation_start"])),
            F.to_date(F.lit(fold["validation_end"])),
        )
    )

    sampled_training_fold = bounded_uniform_sample(
        complete_training_fold,
        LOCAL_TRAIN_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10,
    )
    sampled_validation_fold = bounded_uniform_sample(
        complete_validation_fold,
        LOCAL_VALIDATION_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10 + 1,
    )

    X_train_fold, y_train_fold = spark_vectors_to_csr(
        sampled_training_fold
    )
    X_validation_fold, y_validation_fold = spark_vectors_to_csr(
        sampled_validation_fold
    )

    LOCAL_CV_FOLDS.append(
        {
            "fold": fold_number,
            "X_train": X_train_fold,
            "y_train": y_train_fold,
            "X_validation": X_validation_fold,
            "y_validation": y_validation_fold,
            "scale_pos_weight": positive_class_weight(
                y_train_fold
            ),
        }
    )

    print(
        f"Fold {fold_number}: "
        f"{X_train_fold.shape[0]:,} training rows, "
        f"{X_validation_fold.shape[0]:,} validation rows, "
        f"training delay rate={np.mean(y_train_fold):.4f}, "
        f"validation delay rate={np.mean(y_validation_fold):.4f}"
    )

print("Shared chronological folds prepared for all algorithms.")


## Expanded Hyperparameter Search Spaces

Each algorithm is evaluated with 24 configurations. Across four expanding chronological folds, this produces 96 fitted models per algorithm and 288 fitted models overall.

The search spaces cover:

- Logistic Regression regularization strength and penalty type.
- Random Forest ensemble size, tree depth, leaf size, and feature subsampling.
- XGBoost boosting rounds, learning rate, tree depth, child-weight control, row and feature subsampling, and L2 regularization.

The expanded grids are intended to provide a substantial capstone-scale search. An approximately one-hour runtime is a planning target, not a guarantee: actual duration depends on Databricks hardware, service load, matrix dimensions, and convergence behavior. The notebook does not add artificial waiting time.


In [0]:
LOGISTIC_REGRESSION_PARAMETER_GRID = [
    {"C": 0.01, "penalty": "l2", "solver": "liblinear"},
    {"C": 0.10, "penalty": "l2", "solver": "liblinear"},
    {"C": 1.00, "penalty": "l2", "solver": "liblinear"},
    {"C": 10.0, "penalty": "l2", "solver": "liblinear"},
    {"C": 0.01, "penalty": "l1", "solver": "liblinear"},
    {"C": 0.10, "penalty": "l1", "solver": "liblinear"},
    {"C": 1.00, "penalty": "l1", "solver": "liblinear"},
    {"C": 10.0, "penalty": "l1", "solver": "liblinear"},
]

RANDOM_FOREST_PARAMETER_GRID = [
    {
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 100,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "log2",
    },
    {
        "n_estimators": 300,
        "max_depth": 16,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": "log2",
    },
]

XGBOOST_PARAMETER_GRID = [
    {
        "n_estimators": 100,
        "max_depth": 4,
        "learning_rate": 0.10,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.10,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.80,
        "colsample_bytree": 0.90,
        "reg_lambda": 2.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 2.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 8,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.03,
        "min_child_weight": 7,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
    },
]

# Expanded, reproducible search spaces. Each algorithm receives 24 candidates
# and therefore 96 chronological fits across the four shared folds. Runtime
# depends on the Databricks hardware and current service load; one hour is a
# planning target rather than a guaranteed duration.
LOGISTIC_REGRESSION_PARAMETER_GRID = list(ParameterGrid([
    {
        "C": [0.01, 0.10, 1.0, 10.0],
        "penalty": ["l1"],
        "solver": ["liblinear", "saga"],
    },
    {
        "C": [0.01, 0.10, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["liblinear", "saga", "lbfgs", "newton-cg"],
    },
]))

RANDOM_FOREST_PARAMETER_GRID = list(ParameterGrid({
    "n_estimators": [200, 400, 600],
    "max_depth": [10, 18],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", "log2"],
}))

XGBOOST_PARAMETER_GRID = list(ParameterGrid({
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 7],
    "learning_rate": [0.03, 0.08],
    "min_child_weight": [1, 5],
    "subsample": [0.85],
    "colsample_bytree": [0.85],
    "reg_lambda": [1.0],
}))

print(
    "Candidate configurations: "
    f"LR={len(LOGISTIC_REGRESSION_PARAMETER_GRID)}, "
    f"RF={len(RANDOM_FOREST_PARAMETER_GRID)}, "
    f"XGBoost={len(XGBOOST_PARAMETER_GRID)}"
)
print(
    "Expected chronological fits: "
    f"LR={len(LOGISTIC_REGRESSION_PARAMETER_GRID) * len(TUNING_FOLDS)}, "
    f"RF={len(RANDOM_FOREST_PARAMETER_GRID) * len(TUNING_FOLDS)}, "
    f"XGBoost={len(XGBOOST_PARAMETER_GRID) * len(TUNING_FOLDS)}"
)
print(
    "Total expected fits: "
    f"{(len(LOGISTIC_REGRESSION_PARAMETER_GRID) + len(RANDOM_FOREST_PARAMETER_GRID) + len(XGBOOST_PARAMETER_GRID)) * len(TUNING_FOLDS)}"
)


## Shared Estimator Builders

Each builder creates a fresh standard-Python classifier. Logistic Regression and Random Forest use built-in balanced class weights. XGBoost receives a fold-specific positive-class weight from the shared tuning function.


In [0]:
def build_logistic_regression(params, *, scale_pos_weight=None):
    return LogisticRegression(
        **params,
        class_weight="balanced",
        max_iter=500,
        random_state=RANDOM_SEED,
    )


def build_random_forest(params, *, scale_pos_weight=None):
    return RandomForestClassifier(
        **params,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def build_xgboost(params, *, scale_pos_weight):
    return XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


print("Shared standard-Python estimator builders created.")


## Shared Hyperparameter-Tuning Function

The same function evaluates every algorithm. For each candidate configuration, it:

1. Uses the same four precomputed chronological folds.
2. Fits only on the fold's earlier training observations.
3. Applies training-only class-imbalance controls.
4. Evaluates on the unchanged later validation observations.
5. Averages all metrics across folds.

Results are ranked by delayed-flight Recall, delayed-flight F1-score, PR AUC, ROC AUC, and then training efficiency.


In [0]:
RESULT_COLUMNS = [
    "MODEL",
    "PARAMETERS",
    "ACCURACY",
    "PRECISION",
    "RECALL",
    "F1_SCORE",
    "ROC_AUC",
    "PR_AUC",
    "DELAY_PRECISION",
    "DELAY_RECALL",
    "DELAY_F1",
    "TRAINING_SECONDS",
]


def tune_python_classifier(
    model_name,
    estimator_builder,
    parameter_grid,
    *,
    local_folds=LOCAL_CV_FOLDS,
):
    """Tune one standard-Python classifier on shared time folds."""
    if not parameter_grid:
        raise ValueError(f"{model_name} parameter grid is empty.")

    result_rows = []

    for configuration_number, params in enumerate(
        parameter_grid,
        start=1,
    ):
        fold_metrics = []

        print(
            f"Evaluating {model_name} configuration "
            f"{configuration_number}/{len(parameter_grid)}: {params}"
        )

        for fold in local_folds:
            estimator = estimator_builder(
                params,
                scale_pos_weight=fold["scale_pos_weight"],
            )

            training_start = time.perf_counter()
            estimator.fit(
                fold["X_train"],
                fold["y_train"],
            )
            training_seconds = time.perf_counter() - training_start

            metrics = evaluate_python_classifier(
                estimator,
                fold["X_validation"],
                fold["y_validation"],
            )
            metrics["TRAINING_SECONDS"] = float(training_seconds)
            fold_metrics.append(metrics)

            print(
                f"  Fold {fold['fold']}: "
                f"delay recall={metrics['DELAY_RECALL']:.4f}, "
                f"delay F1={metrics['DELAY_F1']:.4f}, "
                f"PR AUC={metrics['PR_AUC']:.4f}"
            )

        averaged = {
            metric_name: round(
                float(
                    np.mean(
                        [
                            row[metric_name]
                            for row in fold_metrics
                        ]
                    )
                ),
                2 if metric_name == "TRAINING_SECONDS" else 4,
            )
            for metric_name in RESULT_COLUMNS[2:]
        }

        result_rows.append(
            {
                "MODEL": model_name,
                "PARAMETERS": str(params),
                **averaged,
            }
        )

    return (
        spark.createDataFrame(result_rows)
        .select(*RESULT_COLUMNS)
        .orderBy(
            F.desc("DELAY_RECALL"),
            F.desc("DELAY_F1"),
            F.desc("PR_AUC"),
            F.desc("ROC_AUC"),
            F.asc("TRAINING_SECONDS"),
        )
    )


print("Unified standard-Python tuning function created.")


## Tune Logistic Regression

Twenty-four scikit-learn Logistic Regression configurations are evaluated using the shared chronological folds. The search compares L1 and L2 penalties, four regularization strengths, and compatible optimization solvers. Balanced class weights are learned from each training fold.


In [0]:
lr_tuning_results = tune_python_classifier(
    model_name="Logistic Regression",
    estimator_builder=build_logistic_regression,
    parameter_grid=LOGISTIC_REGRESSION_PARAMETER_GRID,
)

display(lr_tuning_results)


## Tune Random Forest

Twenty-four scikit-learn Random Forest configurations are evaluated using the same matrices and chronological folds. The search varies the number of trees, maximum tree depth, minimum leaf size, and feature subsampling strategy. Balanced subsample weights are recalculated for every tree.


In [0]:
rf_tuning_results = tune_python_classifier(
    model_name="Random Forest",
    estimator_builder=build_random_forest,
    parameter_grid=RANDOM_FOREST_PARAMETER_GRID,
)

display(rf_tuning_results)


## Tune XGBoost

Twenty-four XGBoost configurations are evaluated using the same matrices and chronological folds. The search varies boosting rounds, tree depth, learning rate, and minimum child weight. The fold-specific `scale_pos_weight` value is calculated only from that fold's training labels; validation labels never affect class weighting.


In [0]:
xgb_tuning_results = tune_python_classifier(
    model_name="XGBoost",
    estimator_builder=build_xgboost,
    parameter_grid=XGBOOST_PARAMETER_GRID,
)

display(xgb_tuning_results)


## Tuned Model Comparison

The best configuration from each algorithm is selected under one operational ranking:

1. Highest delayed-flight Recall
2. Highest delayed-flight F1-score
3. Highest PR AUC
4. Highest ROC AUC
5. Lowest average training time

Recall receives primary emphasis because a false negative represents an actual delayed flight that the system failed to flag. The remaining metrics prevent the decision from being described as an Accuracy-only comparison and provide evidence about false-positive control, ranking quality, and efficiency.

Delayed-flight Recall is calculated as `TP / (TP + FN)`. It answers: *Of all flights that actually became delayed, what proportion did the model identify?* It is operationally important because a missed delayed flight cannot be prioritized before departure.

Recall is not sufficient by itself. A model could obtain high Recall by flagging too many on-time flights as delayed. Delayed-flight F1-score therefore checks the balance between Recall and Precision, while PR AUC evaluates how effectively the model ranks the minority delayed-flight class across decision thresholds. If the project later defines an explicit acceptable missed-delay rate, an even stronger policy would be to require that minimum Recall first and then select the eligible model with the highest delayed-flight F1-score and PR AUC.


In [0]:
def best_tuned_configuration(results):
    return (
        results
        .orderBy(
            F.desc("DELAY_RECALL"),
            F.desc("DELAY_F1"),
            F.desc("PR_AUC"),
            F.desc("ROC_AUC"),
            F.asc("TRAINING_SECONDS"),
        )
        .limit(1)
    )


best_logistic_regression = best_tuned_configuration(
    lr_tuning_results
)
best_random_forest = best_tuned_configuration(
    rf_tuning_results
)
best_xgboost = best_tuned_configuration(
    xgb_tuning_results
)

tuned_model_comparison = (
    best_logistic_regression
    .unionByName(best_random_forest)
    .unionByName(best_xgboost)
)

ranked_model_comparison = tuned_model_comparison.orderBy(
    F.desc("DELAY_RECALL"),
    F.desc("DELAY_F1"),
    F.desc("PR_AUC"),
    F.desc("ROC_AUC"),
    F.asc("TRAINING_SECONDS"),
)

display(ranked_model_comparison)


## Tuned Confusion Matrices Across Chronological Folds

For each algorithm, its strongest hyperparameter configuration is refitted separately within each of the four expanding-window folds. Predictions from the four validation months are then pooled into one confusion matrix.

These matrices provide an interpretable after-tuning comparison on identical chronological validation observations. They are diagnostic cross-validation summaries, not final holdout results. Notebook 08 remains responsible for definitive evaluation on later untouched data.


In [0]:
best_parameter_rows = {
    "Logistic Regression": best_logistic_regression.first(),
    "Random Forest": best_random_forest.first(),
    "XGBoost": best_xgboost.first(),
}

estimator_builders = {
    "Logistic Regression": build_logistic_regression,
    "Random Forest": build_random_forest,
    "XGBoost": build_xgboost,
}

tuned_confusion_matrices = {}

for model_name, result_row in best_parameter_rows.items():
    if result_row is None:
        raise ValueError(f"No tuned result found for {model_name}.")

    best_parameters = ast.literal_eval(result_row["PARAMETERS"])
    pooled_actual = []
    pooled_predicted = []

    for fold in LOCAL_CV_FOLDS:
        estimator = estimator_builders[model_name](
            best_parameters,
            scale_pos_weight=fold["scale_pos_weight"],
        )
        estimator.fit(fold["X_train"], fold["y_train"])

        pooled_actual.append(fold["y_validation"])
        pooled_predicted.append(
            estimator.predict(fold["X_validation"]).astype(np.int8)
        )

    pooled_actual = np.concatenate(pooled_actual)
    pooled_predicted = np.concatenate(pooled_predicted)

    tuned_confusion_matrices[model_name] = confusion_matrix_table(
        pooled_actual,
        pooled_predicted,
    )

    print(f"Tuned chronological confusion matrix: {model_name}")
    display(tuned_confusion_matrices[model_name])
    print(f"Tuned chronological TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            pooled_actual,
            pooled_predicted,
        )
    )


## Select the Candidate Final Model

The candidate final model is selected dynamically from the ranked comparison. It is not hard-coded to a particular algorithm. Because every candidate used the same feature matrices, chronological folds, validation distributions, class-imbalance policy, and metrics, the decision is not biased by different evaluation datasets.

The selected algorithm and its standard-Python hyperparameters are saved for downstream evaluation.


In [0]:
selected_model_row = ranked_model_comparison.first()

if selected_model_row is None:
    raise ValueError(
        "The ranked tuned-model comparison contains no results."
    )

SELECTED_MODEL_NAME = selected_model_row["MODEL"]
SELECTED_MODEL_PARAMETERS = ast.literal_eval(
    selected_model_row["PARAMETERS"]
)

selected_model_summary = (
    ranked_model_comparison
    .filter(F.col("MODEL") == SELECTED_MODEL_NAME)
    .limit(1)
)

display(selected_model_summary)
print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")
print(
    "Selection policy: highest delayed-flight Recall, followed by "
    "delayed-flight F1-score, PR AUC, ROC AUC, and training efficiency."
)


In [0]:
test

## Candidate-Selection Interpretation

The displayed results determine the selected candidate. The interpretation should be written from the newly generated table after the revised notebook has run; earlier numerical conclusions from the Spark-based workflow must not be reused.

The selected candidate must still be retrained using its chosen standard-Python hyperparameters and evaluated on the untouched final holdout period in Notebook 08. The final holdout results—not the tuning-fold averages—provide the definitive estimate of future predictive performance.


## Persist Modeling Checkpoints

This section saves the datasets and metadata required by the downstream model-evaluation and explainability notebooks.

The historical and hashed training, validation, and test tables retain their existing names and schemas. The tuned-model comparison table also retains its existing schema so that downstream reporting can continue to read the candidate metrics.

The feature-manifest and candidate-selection JSON files record:

- The ordered model-input columns
- The feature-hashing configuration
- The dynamically selected algorithm
- The selected standard-Python hyperparameters
- The chronological tuning process as the selection source

No fitted estimator is saved in this notebook. Notebook 08 must reconstruct the selected standard-Python estimator from the saved algorithm name and parameters, retrain it, calibrate its operational threshold using the September–October validation period, and evaluate it on the untouched November–December holdout.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

import json

from utils.model_training import (
    build_feature_manifest,
    create_feature_hasher,
    hash_modeling_frame,
    prepare_hist_modeling_frame,
    validate_feature_hasher,
)

# Persist only the stable modeling schema without intermediate prior-count columns.
df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

# Rebuild hashed checkpoints from the cleaned hist tables before saving.
feature_hasher = create_feature_hasher()
validate_feature_hasher(feature_hasher)

df_train_hashed = hash_modeling_frame(df_train_hist, feature_hasher)
df_validation_hashed = hash_modeling_frame(df_validation_hist, feature_hasher)
df_test_hashed = hash_modeling_frame(df_test_hist, feature_hasher)

checkpoint_tables = [
    (cfg.MODELING_TRAIN_HASHED_TABLE, df_train_hashed),
    (cfg.MODELING_VALIDATION_HASHED_TABLE, df_validation_hashed),
    (cfg.MODELING_TEST_HASHED_TABLE, df_test_hashed),
    (cfg.MODELING_TRAIN_HIST_TABLE, df_train_hist),
    (cfg.MODELING_VALIDATION_HIST_TABLE, df_validation_hist),
    (cfg.MODELING_TEST_HIST_TABLE, df_test_hist),
]

for table_name, dataframe in checkpoint_tables:
    row_count = dataframe.count()
    print(f"Saving {table_name}: {row_count:,} rows")
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

(
    tuned_model_comparison.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.TUNED_MODEL_COMPARISON_TABLE)
)

feature_manifest = build_feature_manifest(
    selected_model_name=SELECTED_MODEL_NAME,
    selected_model_parameters=SELECTED_MODEL_PARAMETERS,
)

candidate_selection = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selection_source": "Average performance across chronological tuning folds",
    "selected_after_hyperparameter_tuning": True,
    "modeling_implementation": "standard_python",
    "cross_validation": "four expanding chronological folds",
    "class_imbalance_strategy": "training-fold class weights",
}

dbutils.fs.put(
    cfg.MODEL_FEATURE_MANIFEST_PATH,
    json.dumps(feature_manifest, indent=4),
    overwrite=True,
)

dbutils.fs.put(
    cfg.CANDIDATE_SELECTION_PATH,
    json.dumps(candidate_selection, indent=4),
    overwrite=True,
)

print("Modeling checkpoints saved successfully.")
print(f"Feature manifest: {cfg.MODEL_FEATURE_MANIFEST_PATH}")
print(f"Candidate selection: {cfg.CANDIDATE_SELECTION_PATH}")
print(f"Manifest model inputs: {feature_manifest['model_input_columns']}")


In [0]:
import json


candidate_selection_saved = json.loads(
    dbutils.fs.head(
        cfg.CANDIDATE_SELECTION_PATH,
        10_000,
    )
)

print(
    json.dumps(
        candidate_selection_saved,
        indent=4,
    )
)

assert (
    candidate_selection_saved["selected_model_name"]
    == SELECTED_MODEL_NAME
)

assert (
    candidate_selection_saved[
        "selected_model_parameters"
    ]
    == SELECTED_MODEL_PARAMETERS
)

print("Candidate-selection metadata verified.")